# CrackFusion-Tiny: completed-run evaluation

Run all cells on a **CPU runtime**. This reads notebook 03's saved predictions and histories; it does not train or require `src/`. Results go to the selected run's `evaluation/` folder in Drive. The same notebook also runs locally from the repository.

These are **development validation results** from 720 tiles, selected using validation macro F1. Parent-photo separation is unverified. Seed standard deviations describe optimization variation on the same images, not confidence intervals or independent test sets.


In [ ]:
import csv, hashlib, io, json, os, shutil, sys, tempfile, zipfile
from pathlib import Path, PurePosixPath
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve)

RUN_NAME = "run_4d1bf234e5ee01bc"
EXAMPLE_SEED = 42  # Fixed in advance, rather than choosing the best-performing seed.
if "google.colab" in sys.modules:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/Fridge/major_project_dataset")
else:
    BASE = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/crackfusion_tiny.py").is_file())
RUN = BASE / "outputs/crackfusion_tiny" / RUN_NAME
OUT = RUN / "evaluation"
if not (RUN / "completed_seeds.json").is_file():
    raise FileNotFoundError(f"Completed run not found: {RUN}. Finish notebook 03 or set RUN_NAME.")
config = json.loads((RUN / "configuration.json").read_text())
seeds = config["SEEDS"]
assert seeds and len(seeds) == len(set(seeds))
assert sorted(json.loads((RUN / "completed_seeds.json").read_text())) == sorted(seeds), "Some seeds are incomplete"
assert config["PROTOCOL"] == "development", "This notebook evaluates the development validation protocol"
assert hashlib.sha256((RUN / "manifest.csv").read_bytes()).hexdigest() == config["manifest_sha256"], "Manifest hash mismatch"
threshold = float(config["THRESHOLD"])
assert 0 <= threshold <= 1
manifest = pd.read_csv(RUN / "manifest.csv", keep_default_na=False)
assert not manifest.path.duplicated().any()
validation = manifest[manifest.split == "dev_validation"].set_index("path")
assert len(validation) and (validation.split_basis == "tile_development_only").all()
OUT.mkdir(parents=True, exist_ok=True)
print(f"Run: {RUN_NAME}; seeds: {seeds}; validation images per seed: {len(validation)}")
print("Development validation only; parent-photo separation is unverified.")

# Write each artifact completely before replacing its previous copy.
def save_bytes(name, data):
    fd, temp = tempfile.mkstemp(prefix=".evaluation_", dir=OUT)
    try:
        with os.fdopen(fd, "wb") as f:
            f.write(data)
        os.replace(temp, OUT / name)
    finally:
        if os.path.exists(temp): os.unlink(temp)

def save_json(name, data):
    save_bytes(name, (json.dumps(data, indent=2, allow_nan=False) + "\n").encode())

def save_table(name, table):
    save_bytes(name, table.to_csv(index=False).encode())

def save_figure(name, fig):
    fig.tight_layout()
    for extension in ("png", "pdf"):
        buffer = io.BytesIO()
        fig.savefig(buffer, format=extension, dpi=180, bbox_inches="tight")
        save_bytes(f"{name}.{extension}", buffer.getvalue())
    plt.show()
    plt.close(fig)


In [ ]:
def binary_metrics(labels, probabilities, threshold=0.5):
    labels, probabilities = np.asarray(labels).reshape(-1), np.asarray(probabilities).reshape(-1)
    predictions = probabilities >= threshold
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    both = len(np.unique(labels)) == 2
    return dict(accuracy=float(accuracy_score(labels, predictions)),
        balanced_accuracy=float(balanced_accuracy_score(labels, predictions)),
        precision=float(precision_score(labels, predictions, zero_division=0)),
        recall=float(recall_score(labels, predictions, zero_division=0)),
        specificity=float(tn / (tn + fp)) if tn + fp else None,
        f1=float(f1_score(labels, predictions, zero_division=0)),
        macro_f1=float(f1_score(labels, predictions, labels=[0, 1], average='macro', zero_division=0)),
        roc_auc=float(roc_auc_score(labels, probabilities)) if both else None,
        average_precision=float(average_precision_score(labels, probabilities)) if both else None,
        confusion_matrix=[[int(tn), int(fp)], [int(fn), int(tp)]])

def check_predictions(frame):
    assert len(frame) == len(validation) and not frame.path.duplicated().any(), "Missing or duplicate predictions"
    assert set(frame.path) == set(validation.index), "Predictions are not the validation set"
    aligned = validation.loc[frame.path]
    for field in ("label", "surface", "parent_id"):
        assert np.array_equal(frame[field].astype(str).to_numpy(), aligned[field].astype(str).to_numpy()), f"{field} mismatch"
    probabilities = frame.probability.to_numpy(dtype=float)
    assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all(), "Invalid probabilities"
    assert np.array_equal(frame.prediction.to_numpy(), (probabilities >= threshold).astype(int)), "Threshold mismatch"

def compare_metrics(actual, saved):
    for key, value in actual.items():
        assert (value is None and saved[key] is None) or (value is not None and np.allclose(value, saved[key], atol=1e-8)), f"Metric mismatch: {key}"

predictions, histories, metrics = {}, {}, {}
rows, surfaces = [], []
for seed in seeds:
    frame = pd.read_csv(RUN / f"predictions_seed_{seed}.csv", keep_default_na=False)
    check_predictions(frame)
    saved = json.loads((RUN / f"metrics_seed_{seed}.json").read_text())
    assert saved["seed"] == seed and saved["split"] == "validation" and saved["protocol"] == "development"
    metric = binary_metrics(frame.label, frame.probability, threshold)
    compare_metrics(metric, saved["metrics"])
    history = json.loads((RUN / f"history_seed_{seed}.json").read_text())
    assert history and [h["epoch"] for h in history] == list(range(1, len(history) + 1))
    best = max(history, key=lambda h: h["macro_f1"])
    assert np.isclose(best["macro_f1"], metric["macro_f1"]), "Selected checkpoint does not match history"
    rows.append(dict(seed=seed, selected_epoch=best["epoch"], stopping_epoch=history[-1]["epoch"],
                     **{k:v for k,v in metric.items() if k != "confusion_matrix"}))
    for surface, group in frame.groupby("surface"):
        values = binary_metrics(group.label, group.probability, threshold)
        compare_metrics(values, saved["per_surface"][surface])
        surfaces.append(dict(seed=seed, surface=surface, n=len(group), **{k:v for k,v in values.items() if k != "confusion_matrix"}))
    predictions[seed], histories[seed], metrics[seed] = frame, history, metric
    save_table(f"errors_seed_{seed}.csv", frame[frame.label != frame.prediction])
per_seed, per_surface = pd.DataFrame(rows), pd.DataFrame(surfaces)
metric_names = list(metrics[seeds[0]])[:-1]
aggregate = {k:dict(mean=float(per_seed[k].mean()), seed_std=float(per_seed[k].std(ddof=1)) if len(seeds)>1 else None) for k in metric_names}
for key, saved in json.loads((RUN / "summary.json").read_text()).items():
    compare_metrics(aggregate[key], saved)
save_table("metrics_per_seed.csv", per_seed)
save_table("metrics_per_surface.csv", per_surface)
save_table("metrics_mean_seed_std.csv", pd.DataFrame(aggregate).T.rename_axis("metric").reset_index())
display(per_seed.round(4))
display(pd.DataFrame(aggregate).T.round(4))
print("All saved overall, surface and aggregate metrics reproduced from predictions.")


## Learning curves and confusion matrices

The dashed line marks the epoch selected by validation macro F1. Matrix rows are actual labels; columns are predicted labels.


In [ ]:
fig, axes = plt.subplots(len(seeds), 2, figsize=(11, 3 * len(seeds)), squeeze=False)
for i, seed in enumerate(seeds):
    h = pd.DataFrame(histories[seed])
    selected = int(per_seed.set_index("seed").loc[seed, "selected_epoch"])
    axes[i,0].plot(h.epoch, h.train_loss, label="Train")
    axes[i,0].plot(h.epoch, h.validation_loss, label="Validation")
    axes[i,1].plot(h.epoch, h.macro_f1, label="Validation macro F1")
    for ax in axes[i]:
        ax.axvline(selected, color="gray", linestyle="--", label=f"Selected epoch {selected}")
        ax.set_xlabel("Epoch"); ax.set_title(f"Seed {seed}"); ax.legend(fontsize=8)
    axes[i,0].set_ylabel("BCE loss"); axes[i,1].set_ylabel("Macro F1"); axes[i,1].set_ylim(0,1)
save_figure("learning_curves", fig)
fig, axes = plt.subplots(1,len(seeds),figsize=(4*len(seeds),4),squeeze=False)
for ax, seed in zip(axes[0], seeds):
    matrix = np.array(metrics[seed]["confusion_matrix"])
    ax.imshow(matrix, cmap="Blues", vmin=0, vmax=len(validation)/2)
    for (r,c), value in np.ndenumerate(matrix): ax.text(c,r,str(value),ha="center",va="center")
    ax.set(xticks=[0,1], yticks=[0,1], xticklabels=["Non-crack","Crack"], yticklabels=["Non-crack","Crack"],
           xlabel="Predicted", ylabel="Actual", title=f"Seed {seed}; threshold {threshold:g}")
save_figure("confusion_matrices", fig)


## ROC, precision–recall and surface performance

Each curve evaluates the same 720 validation images under one seed. Surface error bars show sample standard deviation across seeds.


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(11,4))
for seed in seeds:
    frame = predictions[seed]
    fpr,tpr,_ = roc_curve(frame.label, frame.probability)
    precision,recall,_ = precision_recall_curve(frame.label, frame.probability)
    axes[0].plot(fpr,tpr,label=f"Seed {seed}: AUC {metrics[seed]['roc_auc']:.3f}")
    axes[1].plot(recall,precision,label=f"Seed {seed}: AP {metrics[seed]['average_precision']:.3f}")
axes[0].plot([0,1],[0,1],"k--",alpha=.4)
axes[1].axhline(predictions[seeds[0]].label.mean(),color="gray",linestyle="--",label="Class prevalence")
for ax in axes: ax.set(xlim=(0,1),ylim=(0,1)); ax.legend(fontsize=8)
axes[0].set(xlabel="False positive rate",ylabel="True positive rate",title="Validation ROC")
axes[1].set(xlabel="Recall",ylabel="Precision",title="Validation precision–recall")
save_figure("roc_precision_recall", fig)
fig, axes = plt.subplots(1,2,figsize=(10,4))
for ax,key in zip(axes,["macro_f1","recall"]):
    grouped = per_surface.groupby("surface")[key]
    means, spread = grouped.mean(), grouped.std(ddof=1).fillna(0)
    ax.bar(means.index,means,yerr=spread,capsize=5,color=["#527fa3","#759b78","#bd9963"])
    ax.set(ylim=(0,1),ylabel=key.replace("_"," "),title="Development validation by surface")
save_figure("surface_performance", fig)


## Error examples

Seed 42 is fixed in advance. These show the eight most confident false positives and false negatives, plus eight correct predictions in manifest order. They are diagnostic examples, not representative prevalence estimates. Images are read directly from the portable bundle; no extraction or retraining is needed.


In [ ]:
assert EXAMPLE_SEED in seeds
bundle = next((p for p in (BASE / "crackfusion_colab_bundle.zip", BASE / "outputs/data_audit/crackfusion_colab_bundle.zip") if p.is_file()), None)
assert bundle is not None, "Place crackfusion_colab_bundle.zip in the dataset folder or outputs/data_audit"
# Copy the small bundle once to local scratch to avoid repeated Drive reads.
with tempfile.TemporaryDirectory() as scratch:
    local_bundle = Path(scratch) / "bundle.zip"
    shutil.copyfile(bundle, local_bundle)
    with zipfile.ZipFile(local_bundle) as archive:
        frame = predictions[EXAMPLE_SEED]
        groups = {
            "false_positives": frame[(frame.label==0)&(frame.prediction==1)].sort_values("probability",ascending=False),
            "false_negatives": frame[(frame.label==1)&(frame.prediction==0)].sort_values("probability"),
            "correct_examples": frame[frame.label==frame.prediction],
        }
        for name, group in groups.items():
            fig, axes = plt.subplots(2,4,figsize=(12,6))
            for ax in axes.flat: ax.axis("off")
            for ax, (_,row) in zip(axes.flat,group.head(8).iterrows()):
                relative = PurePosixPath(row.path)
                assert not relative.is_absolute() and ".." not in relative.parts
                data = archive.read("dataset/SDNET2018/" + row.path)
                assert hashlib.sha256(data).hexdigest() == validation.loc[row.path,"sha256"], f"Image hash mismatch: {row.path}"
                with Image.open(io.BytesIO(data)) as image: ax.imshow(image.convert("RGB"))
                ax.set_title(f"{row.path}\n{row.surface}; true={row.label}, pred={row.prediction}\nP(crack)={row.probability:.3f}",fontsize=8)
            fig.suptitle(f"{name.replace('_',' ').title()} — seed {EXAMPLE_SEED}; threshold {threshold:g}")
            save_figure(name,fig)
report = dict(run=RUN_NAME, seeds=seeds, validation_images_per_seed=len(validation), threshold=threshold,
    protocol="development", split_basis="tile_development_only", manifest_sha256=config["manifest_sha256"],
    example_seed=EXAMPLE_SEED, aggregate=aggregate, selected_epochs=per_seed[["seed","selected_epoch","stopping_epoch"]].to_dict("records"),
    limitation="Validation-selected development results; parent-photo separation unverified. Seed spread is not a confidence interval.",
    artifacts=sorted(p.name for p in OUT.iterdir() if p.is_file() and not p.name.startswith(".") and p.name != "evaluation_summary.json"))
save_json("evaluation_summary.json",report)
print(f"Evaluation complete. Tables, PNG/PDF figures, error lists and summary saved to {OUT}")
